| Model | Type | Dataset 1 (Accuracy/F1/AUROC) | Dataset 2 (Accuracy/F1/AUROC) |
|-------|------|------------------------------|------------------------------|
| [BERT](https://github.com/google-research/bert) | Transformer | 90.2% / 0.85 / 0.76 | 88.1% / 0.82 / 0.73 |
| [RoBERTa](https://github.com/facebookresearch/fairseq/tree/main/examples/roberta) | Transformer | 92.5% / 0.89 / 0.81 | 91.0% / 0.87 / 0.79 |
| [Our Model](link/to/your/repo) | Custom | **94.1%** / **0.91** / **0.85** | **93.2%** / **0.90** / **0.84** |

<table>
  <tr>
    <th>Model</th>
    <th colspan="3" align="center">Dataset 1</th>
    <th colspan="3" align="center">Dataset 2</th>
  </tr>
  <tr>
    <th></th>
    <th>Accuracy</th>
    <th>F1-Score</th>
    <th>AUROC</th>
    <th>Accuracy</th>
    <th>F1-Score</th>
    <th>AUROC</th>
  </tr>
  <tr>
    <td>Model 1</td>
    <td>90.2%</td>
    <td>0.85</td>
    <td>0.76</td>
    <td>88.1%</td>
    <td>0.82</td>
    <td>0.73</td>
  </tr>
  <tr>
    <td>Model 2</td>
    <td>92.5%</td>
    <td>0.89</td>
    <td>0.81</td>
    <td>91.0%</td>
    <td>0.87</td>
    <td>0.79</td>
  </tr>
  <tr>
    <td>Model 3</td>
    <td><b>94.1%</b></td>
    <td><b>0.91</b></td>
    <td><b>0.85</b></td>
    <td><b>93.2%</b></td>
    <td><b>0.90</b></td>
    <td><b>0.84</b></td>
  </tr>
</table>

In [3]:
import tsl
import torch
import numpy as np
import pandas as pd
from tsl.datasets import MetrLA
from einops import rearrange
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.nn import models
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn

import random
import torch
import numpy as np
import os

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler
from pytorch_lightning.profilers import AdvancedProfiler

from torch.optim.lr_scheduler import MultiStepLR
from pytorch_lightning.loggers import TensorBoardLogger



def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

seed_everything(42)

42

In [4]:
dataset = MetrLA(root='./data/metrla')

connectivity = dataset.get_connectivity(threshold=0.1,
                                        include_self=False,
                                        normalize_axis=1,
                                        force_symmetric=False,
                                        layout="edge_index")

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=connectivity,
                                      mask=dataset.mask,
                                      covariates=covariates,
                                      horizon=12,
                                      window=12,
                                      stride=1)
print(torch_dataset)

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:98: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(df.index[0], df.index[-1], freq='5T')
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:109: FutureWarning: The 'method' keyword in DataFrame.replace is deprecated and will be removed in a future version.
  df = df.replace(to_replace=0., method='ffill')


SpatioTemporalDataset(n_samples=34249, n_nodes=207, n_channels=1)


In [6]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=32,
    workers = 4
)

dm.setup()
print(dm)

{Train dataloader: size=24648}
{Validation dataloader: size=2728}
{Test dataloader: size=6849}
{Predict dataloader: None}


In [7]:
from typing import Optional

import torch
from einops import repeat
from torch import Tensor, nn
from torch.nn import functional as F
from torch_geometric.typing import Adj, OptTensor

from tsl.nn.blocks.decoders import MLPDecoder
from tsl.nn.blocks.encoders import TemporalConvNet
from tsl.nn.layers.base import NodeEmbedding
from tsl.nn.layers.graph_convs import DenseGraphConvOrderK, DiffConv
from tsl.nn.layers.norm import Norm
from tsl.nn.models.base_model import BaseModel


class GraphWaveNetModel(BaseModel):
    return_type = Tensor

    def __init__(self,
                 input_size: int,
                 output_size: int,
                 horizon: int,
                 exog_size: int = 0,
                 hidden_size: int = 32,
                 ff_size: int = 256,
                 n_layers: int = 8,
                 temporal_kernel_size: int = 2,
                 spatial_kernel_size: int = 2,
                 learned_adjacency: bool = True,
                 n_nodes: Optional[int] = None,
                 emb_size: int = 10,
                 dilation: int = 2,
                 dilation_mod: int = 2,
                 norm: str = 'batch',
                 dropout: float = 0.3):
        super(GraphWaveNetModel, self).__init__()

        if learned_adjacency:
            assert n_nodes is not None
            self.source_embeddings = NodeEmbedding(n_nodes, emb_size)
            self.target_embeddings = NodeEmbedding(n_nodes, emb_size)
        else:
            self.register_parameter('source_embedding', None)
            self.register_parameter('target_embedding', None)

        self.input_encoder = nn.Linear(input_size + exog_size, hidden_size)

        temporal_conv_blocks = []
        spatial_convs = []
        skip_connections = []
        norms = []
        receptive_field = 1
        for i in range(n_layers):
            d = dilation**(i % dilation_mod)
            temporal_conv_blocks.append(
                TemporalConvNet(input_channels=hidden_size,
                                hidden_channels=hidden_size,
                                kernel_size=temporal_kernel_size,
                                dilation=d,
                                exponential_dilation=False,
                                n_layers=1,
                                causal_padding=False,
                                gated=False))

            spatial_convs.append(
                DiffConv(in_channels=hidden_size,
                         out_channels=hidden_size,
                         k=spatial_kernel_size))

            skip_connections.append(nn.Linear(hidden_size, ff_size))
            norms.append(Norm(norm, hidden_size))
            receptive_field += d * (temporal_kernel_size - 1)
        self.tconvs = nn.ModuleList(temporal_conv_blocks)
        self.sconvs = nn.ModuleList(spatial_convs)
        self.skip_connections = nn.ModuleList(skip_connections)
        self.norms = nn.ModuleList(norms)
        self.dropout = nn.Dropout(dropout)

        self.receptive_field = receptive_field

        dense_sconvs = []
        if learned_adjacency:
            for _ in range(n_layers):
                dense_sconvs.append(
                    DenseGraphConvOrderK(input_size=hidden_size,
                                         output_size=hidden_size,
                                         support_len=1,
                                         order=spatial_kernel_size,
                                         include_self=False,
                                         channel_last=True))
        self.dense_sconvs = nn.ModuleList(dense_sconvs)
        self.readout = nn.Sequential(
            nn.ReLU(),
            MLPDecoder(input_size=ff_size,
                       hidden_size=2 * ff_size,
                       output_size=output_size,
                       horizon=horizon,
                       activation='relu'))

    def get_learned_adj(self):
        logits = F.relu(self.source_embeddings() @ self.target_embeddings().T)
        adj = torch.softmax(logits, dim=1)
        return adj

    def forward(self,
                x: Tensor,
                edge_index: Adj,
                edge_weight: OptTensor = None,
                u: OptTensor = None) -> Tensor:
        """"""
        # x: [b t n f]

        if u is not None:
            if u.dim() == 3:
                u = repeat(u, 'b t f -> b t n f', n=x.size(-2))
            x = torch.cat([x, u], -1)

        if self.receptive_field > x.size(1):
            # pad temporal dimension
            x = F.pad(x, (0, 0, 0, 0, self.receptive_field - x.size(1), 0))

        if len(self.dense_sconvs):
            adj_z = self.get_learned_adj()

        x = self.input_encoder(x)

        out = torch.zeros(1, x.size(1), 1, 1, device=x.device)
        for i, (tconv, sconv, skip_conn, norm) in enumerate(
                zip(self.tconvs, self.sconvs, self.skip_connections,
                    self.norms)):
            res = x
            # temporal conv
            x = tconv(x)
            # residual connection -> out
            out = skip_conn(x) + out[:, -x.size(1):]
            # spatial conv
            xs = sconv(x, edge_index, edge_weight)
            if len(self.dense_sconvs):
                x = xs + self.dense_sconvs[i](x, adj_z)
            else:
                x = xs
            x = self.dropout(x)
            # residual connection -> next layer
            x = x + res[:, -x.size(1):]
            x = norm(x)

        return self.readout(out)


In [8]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
    'mae': torch_metrics.MaskedMAE(),
    'mse': torch_metrics.MaskedMSE(),
    'mae_step_1': torch_metrics.MaskedMAE(at=0),
   'mae_step_2': torch_metrics.MaskedMAE(at=2),
   'mae_step_3': torch_metrics.MaskedMAE(at=5),
   'mae_step_4': torch_metrics.MaskedMAE(at=11)
}

model = GraphWaveNetModel(input_size=1,exog_size=2, hidden_size = 32,
                             output_size=1,spatial_kernel_size=4,
                             horizon=12,n_layers = 8, n_nodes=torch_dataset.n_nodes,
                             learned_adjacency=True)

receptive_field=model.receptive_field

print('receptive_field',model.receptive_field)

def get_model_log_name(model, torch_dataset):
    class_name = model.__class__.__name__
    directed = str(not is_undirected(torch_dataset.edge_index))
    return f"{class_name}_directed_{directed}_RC_{receptive_field}_learned_adjacency_{False}"
    

logger = TensorBoardLogger(
        save_dir=f"logs/{dataset.name}",
        name=get_model_log_name(model,torch_dataset)
)

receptive_field 13


In [9]:
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  'weight_decay':1e-3
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[40, 80, 120]}
)
# 'momentum':0.9,
#                  'nesterov':True

In [10]:
import torch
from pytorch_lightning.callbacks import Callback
import numpy as np

class SimpleGradientMonitor(Callback):
    """Simple callback that monitors gradients to identify vanishing/exploding issues."""
    
    def __init__(self, vanish_threshold=1e-4, explode_threshold=10.0):
        super().__init__()
        self.vanish_threshold = vanish_threshold
        self.explode_threshold = explode_threshold
        
    def on_after_backward(self, trainer, pl_module):
        # Check gradients after backward pass
        for name, param in pl_module.named_parameters():
            if param.grad is not None:
                grad_norm = torch.norm(param.grad).item()
                
                # Log gradient issues
                if grad_norm < self.vanish_threshold:
                    print(f"Vanishing gradient in {name}: {grad_norm:.6f}")
                    
                if grad_norm > self.explode_threshold or np.isnan(grad_norm):
                    print(f"Exploding gradient in {name}: {grad_norm:.6f}")
                    
grad_monitor = SimpleGradientMonitor()

In [11]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f'model_checkpoint/{dataset.name}/{model.__class__.__name__}',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
    verbose=True,
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=30,
        mode='min',
        min_delta = 0.01
    )

trainer = Trainer(
        max_epochs=200,
        limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=[1],
        gradient_clip_val=5,
       callbacks=[early_stop_callback],
      # default_root_dir="logs",
        check_val_every_n_epoch = 5,
        logger=False

    
)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [12]:
trainer.fit(predictor, datamodule=dm)

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /netfs/tsp/student/2022/zhu/ST_RUM/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type              | Params | Mode 
------------------------------------------------------------
0 | loss_fn       | MaskedMAE         | 0      | train
1 | train_metrics | MetricCollection  | 0      | train
2 | val_metrics   | MetricCollection  | 0      | train
3 | test_metrics  | MetricCollection  | 0      | train
4 | model         | GraphWaveNetModel | 333 K  | train
------------------------------------------------------------
333 K     Trainable params
0         Non-trainable params
333 K     Total params
1.335     Total estimated model params size (MB)
163       Modules in train mode
0         Modules in eval mode


Training: |                                                                        | 0/? [00:00<?, ?it/s]

Only args ['u', 'edge_weight', 'x', 'edge_index'] are forwarded to the model (GraphWaveNetModel).
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('train_mae', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('train_mae_step_1', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('train_mae_step_2', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called 

Validation: |                                                                      | 0/? [00:00<?, ?it/s]

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('val_mae', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('val_mae_step_1', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('val_mae_step_2', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('val_mae_step_3', ..., logger=True)` but have no logger configured. You can enable one by doin

...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

`Trainer.fit` stopped: `max_epochs=200` reached.


In [13]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /netfs/tsp/student/2022/zhu/ST_RUM/checkpoints/epoch=199-step=30000.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /netfs/tsp/student/2022/zhu/ST_RUM/checkpoints/epoch=199-step=30000.ckpt


Testing: |                                                                                                    …

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('test_mae', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('test_mae_step_1', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('test_mae_step_2', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/core/module.py:512: You called `self.log('test_mae_step_3', ..., logger=True)` but have no logger configured. You can enable one by 

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    3.0473122596740723     │
│         test_mae          │     3.204998254776001     │
│      test_mae_step_1      │     2.341477870941162     │
│      test_mae_step_2      │    2.8237879276275635     │
│      test_mae_step_3      │    3.2365245819091797     │
│      test_mae_step_4      │    3.7599148750305176     │
│         test_mse          │    40.020442962646484     │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 3.204998254776001,
  'test_mae_step_1': 2.341477870941162,
  'test_mae_step_2': 2.8237879276275635,
  'test_mae_step_3': 3.2365245819091797,
  'test_mae_step_4': 3.7599148750305176,
  'test_mse': 40.020442962646484,
  'test_loss': 3.0473122596740723}]